# XML De-Identification Pipeline (HL7 CDA & Generic XML)

Two-pass approach that works on **any** XML structure:

| Pass | What it does |
|---|---|
| **Pass 1 — Structural rules** | Targets known PHI locations by tag/attribute pattern (dates in `value` attrs, phone in `telecom`, SSN `root`, etc.) |
| **Pass 2 — ZeroShot NER** | Walks every text node in the document, runs the full JSL NLP pipeline, replaces entities found in free text |

Because Pass 2 reads every text node regardless of tag name, it works on CDA, FHIR, custom XML, or any other structure.


In [ ]:
import json, os, sys, re, copy
import xml.etree.ElementTree as ET
from datetime import datetime, date as date_type
import pandas as pd

os.environ["JAVA_HOME"]             = "/opt/homebrew/opt/openjdk@11"
os.environ["PYSPARK_PYTHON"]        = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

with open("spark_jsl.json") as f:
    license_keys = json.load(f)
locals().update(license_keys)
os.environ.update(license_keys)

XML_INPUT  = "file9.txt"
XML_OUTPUT = "file9_deid.xml"
EXCEL_OUT  = "XML_DeID_Results.xlsx"

print("Setup done.")


In [ ]:
# Stop any existing Spark session so the JSL license lock is released
# before starting a new one. This prevents "license already in use" errors.
try:
    from pyspark.sql import SparkSession
    existing = SparkSession.getActiveSession()
    if existing:
        print("Stopping existing Spark session to release license lock...")
        existing.stop()
        import time; time.sleep(3)
        print("Stopped.")
except Exception as e:
    print(f"No existing session to stop: {e}")

import sparknlp, sparknlp_jsl
from sparknlp.base import *
from sparknlp.annotator import *
from sparknlp_jsl.annotator import *
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.ml import Pipeline
import pandas as pd

spark = sparknlp_jsl.start(license_keys["SECRET"])
# Re-import standard library modules overwritten by wildcard imports above
import re, copy, json, os, sys
from datetime import datetime, date as date_type

print(f"Spark NLP version : {sparknlp.version()}")
print(f"Spark NLP JSL     : {sparknlp_jsl.version()}")


## Configuration

In [ ]:
# ── Patient ID lookup ─────────────────────────────────────────────────────
PATIENT_ID_MAP = {
    "Myra Jones"  : "PT-00001",
    # Add more patients here, or load from a database:
    # rows = conn.execute("SELECT full_name, patient_id FROM patients").fetchall()
    # PATIENT_ID_MAP = dict(rows)
}

def get_patient_id(name: str) -> str:
    return PATIENT_ID_MAP.get(name.strip(), f"PT-{abs(hash(name)) % 90000 + 10000}")

# ── Changelog ─────────────────────────────────────────────────────────────
changelog = []

def log(location, field_type, original, rule, deid):
    changelog.append({
        "XML Location"       : location,
        "Field Type"         : field_type,
        "Original Value"     : original,
        "Rule Applied"       : rule,
        "De-identified Value": deid,
    })

print("Config ready.")


## Helper Functions

In [ ]:
# ── Date helpers ──────────────────────────────────────────────────────────

_DATE_FORMATS = [
    "%m/%d/%Y", "%d/%m/%Y", "%Y-%m-%d", "%m-%d-%Y", "%d-%m-%Y",
    "%d.%m.%Y", "%m.%d.%Y", "%Y/%m/%d",
    "%B %d, %Y", "%b %d, %Y", "%d %B %Y", "%d %b %Y",
    "%B %d %Y",  "%b %d %Y",
]

def _parse_date(s):
    s = re.sub(r'(\d+)(st|nd|rd|th)', r'\1', s.strip())
    for fmt in _DATE_FORMATS:
        try: return datetime.strptime(s, fmt)
        except ValueError: pass
    return None

def yyyymmdd_to_month_year(value: str) -> str:
    try:    return datetime.strptime(value[:8], "%Y%m%d").strftime("%B %Y")
    except Exception: pass
    m = re.search(r'\b(19|20)\d{2}\b', value)
    return m.group(0) if m else "[DATE]"

def yyyymmdd_to_age(value: str) -> str:
    try:
        dt = datetime.strptime(value[:8], "%Y%m%d")
        today = date_type.today()
        age = today.year - dt.year - ((today.month, today.day) < (dt.month, dt.day))
        if 0 <= age <= 120:
            return f"{age} years old"
    except Exception:
        pass
    return "[DOB]"

def partial_zip(z: str) -> str:
    z = z.strip()
    return z[:3] + "X" * (len(z) - 3) if len(z) > 3 else z

def iter_text(el):
    """Yield all text fragments inside an element (text + tails of children)."""
    if el.text: yield el.text
    for child in el:
        yield from iter_text(child)
        if child.tail: yield child.tail

# Test
print(yyyymmdd_to_month_year("20120806"))
print(yyyymmdd_to_age("19470501"))
print(partial_zip("97006"))


## Build ZeroShot NER Pipeline

In [ ]:
# ── Build NLP pipeline ────────────────────────────────────────────────────
documentAssembler = DocumentAssembler().setInputCol("text").setOutputCol("document")

splitter = (InternalDocumentSplitter()
    .setInputCols("document").setOutputCol("sentence")
    .setSplitMode("recursive")
    .setSplitPatterns([r"\s+|(?<=\G.{512})"])
    .setPatternsAreRegex(True)
    .setChunkSize(512).setChunkOverlap(50)
    .setEnableSentenceIncrement(True))

tokenizer     = Tokenizer().setInputCols("sentence").setOutputCol("token")
tokenizer_doc = Tokenizer().setInputCols("document").setOutputCol("token_doc")

labels = ["DOCTOR","PATIENT","DATE_OF_BIRTH","DATE","CITY","STREET","STATE",
          "COUNTRY","PHONE","EMAIL","ZIP","USERNAME","ID","BIOID",
          "ORGANIZATION","MEDICAL_RECORD_NUMBER","SSN","AGE"]

zero_shot_ner = (PretrainedZeroShotNERChunker
    .pretrained("zeroshot_ner_deid_subentity_docwise_medium", "en", "clinical/models")
    .setInputCols("sentence").setOutputCol("ner_zero_shot")
    .setPredictionThreshold(0.7).setLabels(labels).setBatchSize(8))

zip_parser      = (ContextualParserModel.pretrained("zip_parser","en","clinical/models")
                   .setInputCols(["document","token_doc"]).setOutputCol("zip_chunks"))
dob_parser      = (ContextualParserModel.pretrained("date_of_birth_parser","en","clinical/models")
                   .setInputCols(["document","token_doc"]).setOutputCol("dob_chunks"))
email_matcher   = (RegexMatcherInternalModel.pretrained("email_matcher","en","clinical/models")
                   .setInputCols(["document"]).setOutputCol("email_chunks"))
country_matcher = (TextMatcherInternalModel.pretrained("country_matcher","en","clinical/models")
                   .setInputCols(["document","token_doc"]).setOutputCol("country_chunks")
                   .setMergeOverlapping(True))

chunk_merge_ner = (ChunkMergeModel()
    .setInputCols("ner_zero_shot").setOutputCol("ner_merged")
    .setMergeOverlapping(True).setSelectionStrategy("DiverseLonger")
    .setResetSentenceIndices(True)
    .setReplaceDict({"DATE_OF_BIRTH":"DOB","MEDICAL_RECORD_NUMBER":"MEDICALRECORD"}))

chunk_merge_rules = (ChunkMergeModel()
    .setInputCols("zip_chunks","email_chunks","dob_chunks","country_chunks")
    .setOutputCol("rules_merged")
    .setMergeOverlapping(True).setResetSentenceIndices(True)
    .setOrderingFeatures(["ChunkBegin"]).setSelectionStrategy("Sequential"))

chunk_merge_final = (ChunkMergeModel()
    .setInputCols("ner_merged","rules_merged").setOutputCol("ner_chunk")
    .setMergeOverlapping(True).setResetSentenceIndices(True)
    .setOrderingFeatures(["ChunkBegin"]).setSelectionStrategy("Sequential"))

nlp_pipeline = Pipeline(stages=[
    documentAssembler, splitter, tokenizer, tokenizer_doc,
    zero_shot_ner, chunk_merge_ner,
    zip_parser, dob_parser, email_matcher, country_matcher,
    chunk_merge_rules, chunk_merge_final
])

nlp_model = nlp_pipeline.fit(spark.createDataFrame([[""]],("text",)))
print("NLP pipeline ready.")


## Custom De-Identification UDF

In [ ]:
# ── Self-contained UDF — all imports and helpers defined inside ───────────
@F.udf(StringType())
def custom_deid_udf(text, begins, ends, results, meta_list):
    import re
    from datetime import datetime, date as date_type

    _PATIENT_ID_MAP = {"Myra Jones": "PT-00001"}
    _DATE_FORMATS = [
        "%m/%d/%Y", "%d/%m/%Y", "%Y-%m-%d", "%m-%d-%Y", "%d-%m-%Y",
        "%d.%m.%Y", "%m.%d.%Y", "%Y/%m/%d",
        "%B %d, %Y", "%b %d, %Y", "%d %B %Y", "%d %b %Y",
        "%B %d %Y",  "%b %d %Y",
    ]

    def _parse_date(s):
        s = re.sub(r'(\d+)(st|nd|rd|th)', r'\1', s.strip())
        for fmt in _DATE_FORMATS:
            try: return datetime.strptime(s, fmt)
            except ValueError: pass
        return None

    def _get_patient_id(name):
        return _PATIENT_ID_MAP.get(name, f"PT-{abs(hash(name)) % 90000 + 10000}")

    def _date_to_month_year(s):
        dt = _parse_date(s)
        if dt: return dt.strftime("%B %Y")
        m = re.search(r'\b(19|20)\d{2}\b', s)
        return m.group(0) if m else "[DATE]"

    def _dob_to_age(s):
        dt = _parse_date(s)
        if dt:
            today = date_type.today()
            age = today.year - dt.year - ((today.month, today.day) < (dt.month, dt.day))
            if 0 <= age <= 120: return f"{age} years old"
        return "[DOB]"

    def _partial_zip(z):
        z = z.strip()
        return z[:3] + "X" * (len(z) - 3) if len(z) > 3 else z

    if not text or not results: return text
    chunks = []
    for i, chunk_text in enumerate(results):
        b      = begins[i]    if begins    else 0
        e      = ends[i]      if ends      else 0
        meta   = meta_list[i] if meta_list else {}
        entity = meta.get("entity", "") if meta else ""
        chunks.append((b, e, entity, chunk_text))

    # Replace from end → start so character positions stay valid
    chunks.sort(key=lambda x: x[0], reverse=True)
    result = text
    for begin, end, entity, chunk_text in chunks:
        if   entity == "DOCTOR":                continue
        elif entity == "PATIENT":               replacement = _get_patient_id(chunk_text)
        elif entity == "DATE":                  replacement = _date_to_month_year(chunk_text)
        elif entity == "DOB":                   replacement = _dob_to_age(chunk_text)
        elif entity in ("ZIP", "ZIPCODE"):      replacement = _partial_zip(chunk_text)
        else:                                   replacement = f"[{entity}]"
        result = result[:begin] + replacement + result[end + 1:]
    return result

print("UDF registered.")


## Pass 1 — Structural XML Rules

Targets known PHI patterns by tag name and attribute regardless of XML schema.
Works for CDA, FHIR, and custom XML formats.


In [ ]:
# ── Register namespaces so output keeps original prefixes ─────────────────
ET.register_namespace("",      "urn:hl7-org:v3")
ET.register_namespace("xsi",   "http://www.w3.org/2001/XMLSchema-instance")
ET.register_namespace("sdtc",  "urn:hl7-org:sdtc")
ET.register_namespace("cda",   "urn:hl7-org:v3")

tree = ET.parse(XML_INPUT)
root = tree.getroot()

# ── Derive namespace from root tag ────────────────────────────────────────
import re as _re
_ns_match = _re.match(r'\{(.+?)\}', root.tag)
NS = _ns_match.group(1) if _ns_match else ""

def t(name):
    return f"{{{NS}}}{name}" if NS else name

# ── SSN root ──────────────────────────────────────────────────────────────
SSN_ROOT = "2.16.840.1.113883.4.1"

# ─────────────────────────────────────────────────────────────────────────
# Structural rules — tag-based
# Works for any XML; tags that don't exist are simply skipped.
# ─────────────────────────────────────────────────────────────────────────

changes = 0

# 1. Patient name (CDA: patient/name/given + family)
for patient_el in root.iter(t("patient")):
    for name_el in patient_el.findall(f".//{t('name')}"):
        given  = " ".join(g.text.strip() for g in name_el.findall(t("given"))  if g.text)
        family = " ".join(f.text.strip() for f in name_el.findall(t("family")) if f.text)
        full   = f"{given} {family}".strip()
        if full:
            pid = get_patient_id(full)
            log("patient/name", "PATIENT_NAME", full, "→ Patient ID", pid)
            for g in name_el.findall(t("given")):  g.text  = ""
            for f in name_el.findall(t("family")): f.text  = pid
            changes += 1

# 2. Doctor / assignedPerson names — keep as-is, just log
for tag in ("assignedPerson", "responsibleParty"):
    for el in root.iter(t(tag)):
        for name_el in el.findall(f".//{t('name')}"):
            given  = " ".join(g.text.strip() for g in name_el.findall(t("given"))  if g.text)
            family = " ".join(f.text.strip() for f in name_el.findall(t("family")) if f.text)
            full   = f"{given} {family}".strip()
            if full:
                log(f"{tag}/name", "DOCTOR_NAME", full, "kept as-is", full)

# 3. Related / associated persons → [RELATED PERSON]
for tag in ("relatedPerson", "associatedPerson", "informationRecipient"):
    for el in root.iter(t(tag)):
        for name_el in el.findall(f".//{t('name')}"):
            given  = " ".join(g.text.strip() for g in name_el.findall(t("given"))  if g.text)
            family = " ".join(f.text.strip() for f in name_el.findall(t("family")) if f.text)
            full   = f"{given} {family}".strip()
            if full:
                log(f"{tag}/name", "RELATED_PERSON", full, "→ [RELATED_PERSON]", "[RELATED_PERSON]")
                for g in name_el.findall(t("given")):  g.text  = ""
                for f in name_el.findall(t("family")): f.text  = "[RELATED_PERSON]"
                changes += 1

# 4. Date of birth — birthTime @value
for el in root.iter(t("birthTime")):
    v = el.get("value", "")
    if v:
        age = yyyymmdd_to_age(v)
        log("birthTime/@value", "DOB", v, "→ Age", age)
        el.set("value", age)
        changes += 1

# 5. SSN — <id root="2.16.840.1.113883.4.1">
for el in root.iter(t("id")):
    if el.get("root") == SSN_ROOT:
        ext = el.get("extension", "")
        log("id[@root=SSN]/@extension", "SSN", ext, "→ [SSN]", "[SSN]")
        el.set("extension", "[SSN]")
        changes += 1

# 6. Service dates — effectiveTime / low / high @value (YYYYMMDD)
_svc_date_tags = {"effectiveTime", "low", "high", "time"}
for el in root.iter():
    local = el.tag.split("}")[-1] if "}" in el.tag else el.tag
    if local in _svc_date_tags:
        v = el.get("value", "")
        if v and re.match(r'^(19|20)\d{6}', v):
            my = yyyymmdd_to_month_year(v)
            log(f"{local}/@value", "SERVICE_DATE", v, "→ Month+Year", my)
            el.set("value", my)
            changes += 1

# 7. ZIP / postalCode
for el in root.iter(t("postalCode")):
    if el.text and el.text.strip():
        z = el.text.strip()
        pz = partial_zip(z)
        log("postalCode", "ZIP", z, "→ partial mask", pz)
        el.text = pz
        changes += 1

# 8. Phone — telecom @value containing tel:
for el in root.iter(t("telecom")):
    v = el.get("value", "")
    if v.startswith("tel:"):
        log("telecom/@value", "PHONE", v, "→ tel:[PHONE]", "tel:[PHONE]")
        el.set("value", "tel:[PHONE]")
        changes += 1

# 9. Street address
for el in root.iter(t("streetAddressLine")):
    if el.text and el.text.strip():
        log("streetAddressLine", "STREET", el.text.strip(), "→ [STREET]", "[STREET]")
        el.text = "[STREET]"
        changes += 1

print(f"Pass 1 (structural rules) complete — {changes} changes.")


## Pass 2 — ZeroShot NER on All Text Nodes

Collects every text node from the XML tree and runs the full JSL NLP pipeline.
This catches any PHI in narrative sections, table cells, comments, or custom elements
regardless of the XML schema used.


In [ ]:
# ── Collect every non-empty text node in the document ─────────────────────
# Each entry: (element, attr — 'text' or 'tail', original text)
text_nodes = []
for el in root.iter():
    if el.text and el.text.strip():
        text_nodes.append((el, "text", el.text))
    if el.tail and el.tail.strip():
        text_nodes.append((el, "tail", el.tail))

print(f"Text nodes collected: {len(text_nodes)}")

# Deduplicate for NLP (process each unique string once)
unique_texts = list({txt for _, _, txt in text_nodes})
print(f"Unique text strings to process: {len(unique_texts)}")

if unique_texts:
    df = spark.createDataFrame(
        list(enumerate(unique_texts)),
        ["idx", "text"]
    )

    ner_out = nlp_model.transform(df)

    result_df = ner_out.withColumn(
        "deid_text",
        custom_deid_udf(
            F.col("text"),
            F.col("ner_chunk.begin"),
            F.col("ner_chunk.end"),
            F.col("ner_chunk.result"),
            F.col("ner_chunk.metadata"),
        )
    )

    # Build lookup: original text → de-identified text
    deid_map = {
        row["text"]: row["deid_text"]
        for row in result_df.select("text", "deid_text").collect()
        if row["text"] != row["deid_text"]
    }

    print(f"Text nodes with PHI detected: {len(deid_map)}")

    # Apply replacements back to the XML tree
    applied = 0
    for el, attr, orig_text in text_nodes:
        deid_text = deid_map.get(orig_text)
        if deid_text:
            tag_name = el.tag.split("}")[-1] if "}" in el.tag else el.tag
            log(f"<{tag_name}> ({attr})", "NLP_TEXT", orig_text[:80], "ZeroShot NER", deid_text[:80])
            if attr == "text":
                el.text = deid_text
            else:
                el.tail = deid_text
            applied += 1

    print(f"Pass 2 (ZeroShot NER) complete — {applied} text nodes updated.")
else:
    print("No text nodes found.")


In [ ]:
# ── Save de-identified XML ────────────────────────────────────────────────
tree.write(XML_OUTPUT, encoding="unicode", xml_declaration=True)
print(f"De-identified XML saved: {XML_OUTPUT}")
print(f"Total changes logged   : {len(changelog)}")

# Stop Spark to release the JSL license lock for the next session
spark.stop()
print("Spark stopped — license released.")


In [ ]:
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

changes_df = pd.DataFrame(changelog)

wb = openpyxl.Workbook()
ws = wb.active
ws.title = "De-ID Changes"

header_fill = PatternFill("solid", fgColor="1F4E79")
header_font = Font(bold=True, color="FFFFFF", size=11)
alt_fill    = PatternFill("solid", fgColor="DEEAF1")
orig_fill   = PatternFill("solid", fgColor="FCE4D6")
deid_fill   = PatternFill("solid", fgColor="E2EFDA")
wrap_align  = Alignment(wrap_text=True, vertical="top")
thin_border = Border(
    left=Side(style="thin"), right=Side(style="thin"),
    top=Side(style="thin"),  bottom=Side(style="thin")
)

headers    = ["#", "XML Location", "Field Type", "Original Value", "Rule Applied", "De-identified Value"]
col_widths = [4,   32,             18,            30,               22,             30]

for ci, (h, w) in enumerate(zip(headers, col_widths), 1):
    cell = ws.cell(row=1, column=ci, value=h)
    cell.font = header_font; cell.fill = header_fill
    cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    cell.border = thin_border
    ws.column_dimensions[get_column_letter(ci)].width = w
ws.row_dimensions[1].height = 22

for ri, row in changes_df.iterrows():
    er = ri + 2
    is_alt = ri % 2 == 0
    vals = [ri+1, row["XML Location"], row["Field Type"],
            row["Original Value"], row["Rule Applied"], row["De-identified Value"]]
    for ci, val in enumerate(vals, 1):
        cell = ws.cell(row=er, column=ci, value=str(val) if val else "")
        cell.alignment = Alignment(horizontal="center", vertical="top") if ci == 1 else wrap_align
        cell.border = thin_border
        if   ci == 4: cell.fill = orig_fill
        elif ci == 6: cell.fill = deid_fill
        elif is_alt:  cell.fill = alt_fill
    ws.row_dimensions[er].height = 45

ws.freeze_panes = "A2"

ws2 = wb.create_sheet("Summary")
ws2.column_dimensions["A"].width = 25
ws2.column_dimensions["B"].width = 45
summary = [
    ("Input file",    XML_INPUT),
    ("Output file",   XML_OUTPUT),
    ("Total changes", str(len(changelog))),
    ("", ""),
    ("Pass 1 — Structural rules", ""),
    ("  PATIENT_NAME",    "→ Patient ID from lookup"),
    ("  DOCTOR_NAME",     "→ kept as-is"),
    ("  RELATED_PERSON",  "→ [RELATED_PERSON]"),
    ("  DOB",             "→ Age (e.g. 78 years old)"),
    ("  SERVICE_DATE",    "→ Month + Year"),
    ("  SSN",             "→ [SSN]"),
    ("  PHONE",           "→ tel:[PHONE]"),
    ("  STREET",          "→ [STREET]"),
    ("  ZIP",             "→ first 3 + XX"),
    ("", ""),
    ("Pass 2 — ZeroShot NER", ""),
    ("  Model", "zeroshot_ner_deid_subentity_docwise_medium"),
    ("  Scope", "All text nodes in the XML document"),
    ("  Entities", "PATIENT DOCTOR DATE DOB ZIP SSN PHONE EMAIL STREET + 9 more"),
]
for r, (k, v) in enumerate(summary, 1):
    ws2.cell(row=r, column=1, value=k).font = Font(bold=bool(k and not k.startswith(" ")))
    ws2.cell(row=r, column=2, value=v)

wb.save(EXCEL_OUT)
print(f"Excel report saved: {EXCEL_OUT}")
changes_df
